<a href="https://colab.research.google.com/github/zhilyaevaviktorija/machine_learning/blob/main/homeworks/Homework_7_%22trees_hw_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание: cравнение древовидных алгоритмов на NLP-данных

**Задача:** oбучить разные древовидные модели на текстах новостей, замерить test accuracy и время обучения, заполнить таблицы.

## 1. Загрузка и подготовка данных (код дан)

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time
import pandas as pd

# Загружаем 3 категории
categories = ['rec.sport.baseball', 'sci.space', 'comp.graphics']
newsgroups = fetch_20newsgroups(subset='all', categories=categories, shuffle=True, random_state=42)

# TF-IDF векторизация
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
X = vectorizer.fit_transform(newsgroups.data)
y = newsgroups.target

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

Размер обучающей выборки: (2363, 5000)
Размер тестовой выборки: (591, 5000)


## 2. Образец: Дерево решений (Decision Tree)

**Гиперпараметр:** `max_depth`

|   max_depth |   test_accuracy |   time_sec |
|------------:|----------------:|-----------:|
|           3 |          0.6244 |       0.14 |
|           5 |          0.6971 |       0.22 |
|          10 |          0.7716 |       0.42 |
|          20 |          0.8409 |       0.53 |
|     **nan** |      **0.8697** |   **0.72** |

In [ ]:
from sklearn.tree import DecisionTreeClassifier

results_tree = []

# Образец: мы делаем перебор значений max_depth = [3, 5, 10, 20, None])
# Для каждого: замер времени, обучение, accuracy на тесте, сохранение в results_tree

for depth in [3, 5, 10, 20, None]:
    start = time.time()
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(X_train, y_train)
    fit_time = time.time() - start
    acc = accuracy_score(y_test, clf.predict(X_test))
    results_tree.append({'max_depth': depth, 'test_accuracy': round(acc, 4), 'time_sec': round(fit_time, 2)})
    print(f"depth={depth}: acc={acc:.4f}, time={fit_time:.2f}s")

df_tree = pd.DataFrame(results_tree)
print("\nТаблица 1. Дерево решений")
print(df_tree.to_markdown(index=False))

depth=3: acc=0.6244, time=0.14s
depth=5: acc=0.6971, time=0.22s
depth=10: acc=0.7716, time=0.42s
depth=20: acc=0.8409, time=0.53s
depth=None: acc=0.8697, time=0.72s

Таблица 1. Дерево решений
|   max_depth |   test_accuracy |   time_sec |
|------------:|----------------:|-----------:|
|           3 |          0.6244 |       0.14 |
|           5 |          0.6971 |       0.22 |
|          10 |          0.7716 |       0.42 |
|          20 |          0.8409 |       0.53 |
|         nan |          0.8697 |       0.72 |


## 3. Заполните таблицы для остальных алгоритмов

По аналогии с образцом обучите следующие модели и запишите результаты.

### 3.1. Случайный лес (Random Forest)

**Гиперпараметр:** `max_depth` (фиксируем `n_estimators=100`)

|   max_depth |   test_accuracy |   time_sec |
|------------:|----------------:|-----------:|
|           5 |          0.9002 |       0.43 |
|          10 |          0.9154 |       0.7  |
|          20 |          0.9357 |       1.18 |
|     **nan** |      **0.9526** |   **2.94** |

In [ ]:
from sklearn.ensemble import RandomForestClassifier

results_rf = []

# Перебор depth
for depth in [5, 10, 20, None]:
  start = time.time()
  clf = RandomForestClassifier(n_estimators=100, max_depth=depth, random_state=42)
  clf.fit(X_train, y_train)
  fit_time = time.time() - start
  acc = accuracy_score(y_test, clf.predict(X_test))
  results_rf.append({'max_depth': depth, 'test_accuracy': round(acc, 4), 'time_sec': round(fit_time, 2)})
  print(f"depth={depth}: acc={acc:.4f}, time={fit_time:.2f}s")

df_rf = pd.DataFrame(results_rf)
print("\nТаблица 2. Случайный лес")
print(df_rf.to_markdown(index=False))

depth=5: acc=0.9002, time=0.43s
depth=10: acc=0.9154, time=0.70s
depth=20: acc=0.9357, time=1.18s
depth=None: acc=0.9526, time=2.94s

Таблица 2. Случайный лес
|   max_depth |   test_accuracy |   time_sec |
|------------:|----------------:|-----------:|
|           5 |          0.9002 |       0.43 |
|          10 |          0.9154 |       0.7  |
|          20 |          0.9357 |       1.18 |
|         nan |          0.9526 |       2.94 |


### 3.2. Градиентный бустинг (Gradient Boosting)

**Гиперпараметр:** `learning_rate` (фиксируем `n_estimators=100, max_depth=3`)

|   learning_rate |   test_accuracy |   time_sec |
|----------------:|----------------:|-----------:|
|            0.01 |          0.8782 |      49.39 |
|            0.05 |          0.9475 |      53.64 |
|            0.1  |          0.9577 |      41.84 |
|        **0.5**  |      **0.9662** |  **41.35** |

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

results_gb = []

# Перебор learning_rate
for learning_rate in [0.01, 0.05, 0.1, 0.5]:
  start = time.time()
  clf = GradientBoostingClassifier(n_estimators=100, learning_rate=learning_rate, max_depth=3, random_state=42)
  clf.fit(X_train, y_train)
  fit_time = time.time() - start
  acc = accuracy_score(y_test, clf.predict(X_test))
  results_gb.append({'learning_rate': learning_rate, 'test_accuracy': round(acc, 4), 'time_sec': round(fit_time, 2)})
  print(f"learning_rate={learning_rate}: acc={acc:.4f}, time={fit_time:.2f}s")

df_gb = pd.DataFrame(results_gb)
print("\nТаблица 3. Градиентный бустинг")
print(df_gb.to_markdown(index=False))

learning_rate=0.01: acc=0.8782, time=49.39s
learning_rate=0.05: acc=0.9475, time=53.64s
learning_rate=0.1: acc=0.9577, time=41.84s
learning_rate=0.5: acc=0.9662, time=41.35s

Таблица 3. Градиентный бустинг
|   learning_rate |   test_accuracy |   time_sec |
|----------------:|----------------:|-----------:|
|            0.01 |          0.8782 |      49.39 |
|            0.05 |          0.9475 |      53.64 |
|            0.1  |          0.9577 |      41.84 |
|            0.5  |          0.9662 |      41.35 |


### 3.3. XGBoost

**Гиперпараметр:** `gamma` (фиксируем `n_estimators=100, learning_rate=0.1`)

|   gamma |   test_accuracy |   time_sec |
|--------:|----------------:|-----------:|
|     0   |          0.9459 |      53.14 |
|     0.1 |          0.9442 |      38.51 |
| **0.5** |      **0.9492** |  **38.07** |
|     1   |          0.9492 |      47.17 |

In [ ]:
from xgboost import XGBClassifier

results_xgb = []

# Перебор gamma, use_label_encoder=False, eval_metric='logloss'
# Параметр use_label_encoder=False не работает в данной версии
for gamma in [0, 0.1, 0.5, 1.0]:
  start = time.time()
  clf = XGBClassifier(n_estimators=100, gamma=gamma, learning_rate=0.1, eval_metric='logloss', random_state=42)
  clf.fit(X_train, y_train)
  fit_time = time.time() - start
  acc = accuracy_score(y_test, clf.predict(X_test))
  results_xgb.append({'gamma': gamma, 'test_accuracy': round(acc, 4), 'time_sec': round(fit_time, 2)})
  print(f"gamma={gamma}: acc={acc:.4f}, time={fit_time:.2f}s")

df_xgb = pd.DataFrame(results_xgb)
print("\nТаблица 4. XGBoost")
print(df_xgb.to_markdown(index=False))

gamma=0: acc=0.9459, time=53.14s
gamma=0.1: acc=0.9442, time=38.51s
gamma=0.5: acc=0.9492, time=38.07s
gamma=1.0: acc=0.9492, time=47.17s

Таблица 4. XGBoost
|   gamma |   test_accuracy |   time_sec |
|--------:|----------------:|-----------:|
|     0   |          0.9459 |      53.14 |
|     0.1 |          0.9442 |      38.51 |
|     0.5 |          0.9492 |      38.07 |
|     1   |          0.9492 |      47.17 |


### 3.4. AdaBoost

**Гиперпараметр:** `learning_rate` (фиксируем `n_estimators=100`)

|   learning_rate |   test_accuracy |   time_sec |
|----------------:|----------------:|-----------:|
|             0.5 |          0.8629 |      10.06 |
|             1   |          0.9205 |       6.18 |
|         **1.5** |      **0.9255** |   **7.17** |
|             2   |          0.8731 |       6.17 |

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

results_ada = []

# Перебор learning_rate
for learning_rate in [0.5, 1.0, 1.5, 2.0]:
  start = time.time()
  clf = AdaBoostClassifier(n_estimators=100, learning_rate=learning_rate, random_state=42)
  clf.fit(X_train, y_train)
  fit_time = time.time() - start
  acc = accuracy_score(y_test, clf.predict(X_test))
  results_ada.append({'learning_rate': learning_rate, 'test_accuracy': round(acc, 4), 'time_sec': round(fit_time, 2)})
  print(f"learning_rate={learning_rate}: acc={acc:.4f}, time={fit_time:.2f}s")

df_ada = pd.DataFrame(results_ada)
print("\nТаблица 5. AdaBoost")
print(df_ada.to_markdown(index=False))

learning_rate=0.5: acc=0.8629, time=10.06s
learning_rate=1.0: acc=0.9205, time=6.18s
learning_rate=1.5: acc=0.9255, time=7.17s
learning_rate=2.0: acc=0.8731, time=6.17s

Таблица 5. AdaBoost
|   learning_rate |   test_accuracy |   time_sec |
|----------------:|----------------:|-----------:|
|             0.5 |          0.8629 |      10.06 |
|             1   |          0.9205 |       6.18 |
|             1.5 |          0.9255 |       7.17 |
|             2   |          0.8731 |       6.17 |


### 3.5. LightGBM

**Гиперпараметр:** `num_leaves` (фиксируем `n_estimators=100, learning_rate=0.1`)

|   num_leaves |   test_accuracy |   time_sec |
|-------------:|----------------:|-----------:|
|       **15** |      **0.9662** |   **5.06** |
|           31 |          0.9594 |      22.52 |
|           63 |          0.9577 |      17.72 |
|          127 |          0.9594 |      25.41 |

In [ ]:
from lightgbm import LGBMClassifier

results_lgbm = []

# Перебор num_leaves, n_jobs=-1
for num_leaves in [15, 31, 63, 127]:
  start = time.time()
  clf = LGBMClassifier(num_leaves=num_leaves, n_estimators=100, learning_rate=0.1, n_jobs=-1, random_state=42)
  clf.fit(X_train, y_train)
  fit_time = time.time() - start
  acc = accuracy_score(y_test, clf.predict(X_test))
  results_lgbm.append({'num_leaves': num_leaves, 'test_accuracy': round(acc, 4), 'time_sec': round(fit_time, 2)})
  print(f"num_leaves={num_leaves}: acc={acc:.4f}, time={fit_time:.2f}s")

df_lgbm = pd.DataFrame(results_lgbm)
print("\nТаблица 6. LightGBM")
print(df_lgbm.to_markdown(index=False))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.113493 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 61063
[LightGBM] [Info] Number of data points in the train set: 2363, number of used features: 2963
[LightGBM] [Info] Start training from score -1.119999
[LightGBM] [Info] Start training from score -1.093126
[LightGBM] [Info] Start training from score -1.083076


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


num_leaves=15: acc=0.9662, time=5.06s
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.067406 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 61063
[LightGBM] [Info] Number of data points in the train set: 2363, number of used features: 2963
[LightGBM] [Info] Start training from score -1.119999
[LightGBM] [Info] Start training from score -1.093126
[LightGBM] [Info] Start training from score -1.083076


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


num_leaves=31: acc=0.9594, time=22.52s
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.071313 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 61063
[LightGBM] [Info] Number of data points in the train set: 2363, number of used features: 2963
[LightGBM] [Info] Start training from score -1.119999
[LightGBM] [Info] Start training from score -1.093126
[LightGBM] [Info] Start training from score -1.083076
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


num_leaves=63: acc=0.9577, time=17.72s
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.071436 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 61063
[LightGBM] [Info] Number of data points in the train set: 2363, number of used features: 2963
[LightGBM] [Info] Start training from score -1.119999
[LightGBM] [Info] Start training from score -1.093126
[LightGBM] [Info] Start training from score -1.083076
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### 3.6. CatBoost

**Гиперпараметр:** `depth` (фиксируем `iterations=100, learning_rate=0.1`)

|   depth |   test_accuracy |   time_sec |
|--------:|----------------:|-----------:|
|       3 |          0.934  |      13.62 |
|       5 |          0.9425 |      35.3  |
|   **7** |      **0.9442** | **122.55** |
|      10 |          0.9526 |     905.16 |

In [ ]:
!pip install catboost -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.2 MB/s eta 0:00:00


In [ ]:
from catboost import CatBoostClassifier

results_cat = []

# Ваш код здесь (перебор depth, verbose=0)
for depth in [3, 5, 7, 10]:
  start = time.time()
  clf = CatBoostClassifier(depth=depth, iterations=100, learning_rate=0.1, verbose=0, random_state=42)
  clf.fit(X_train, y_train)
  fit_time = time.time() - start
  acc = accuracy_score(y_test, clf.predict(X_test))
  results_cat.append({'depth': depth, 'test_accuracy': round(acc, 4), 'time_sec': round(fit_time, 2)})
  print(f"depth={depth}: acc={acc:.4f}, time={fit_time:.2f}s")

df_cat = pd.DataFrame(results_cat)
print("\nТаблица 7. CatBoost")
print(df_cat.to_markdown(index=False))

depth=3: acc=0.9340, time=13.62s
depth=5: acc=0.9425, time=35.30s
depth=7: acc=0.9442, time=122.55s
depth=10: acc=0.9526, time=905.16s

Таблица 7. CatBoost
|   depth |   test_accuracy |   time_sec |
|--------:|----------------:|-----------:|
|       3 |          0.934  |      13.62 |
|       5 |          0.9425 |      35.3  |
|       7 |          0.9442 |     122.55 |
|      10 |          0.9526 |     905.16 |


## 4. Итоговая таблица лучших результатов

In [ ]:
# Соберите лучшие результаты из каждой таблицы вручную или автоматически
# Пример структуры:

summary = pd.DataFrame([
    ('Decision Tree', 'nan', 0.8697, 0.72),
    ('Random Forest', 'nan', 0.9526, 2.94),
    ('Gradient Boosting', 0.5, 0.9662, 41.35),
    ('XGBoost', 0.5, 0.9492, 38.07),
    ('AdaBoost', 1.5, 0.9255, 7.17),
    ('LightGBM', 15, 0.9662, 5.06),
    ('CatBoost', 7, 0.9442, 122.55), # depth=7 показывает лучший результат accuracy=0.9442 при минимальном времени обучения
], columns=['Algorithm', 'Best params', 'Best test acc', 'Time (sec)'])

print("Итоговая таблица лучших результатов")
print(summary.to_markdown(index=False))

Итоговая таблица лучших результатов
| Algorithm         |   Best params |   Best test acc |   Time (sec) |
|:------------------|--------------:|----------------:|-------------:|
| Decision Tree     |         nan   |          0.8697 |         0.72 |
| Random Forest     |         nan   |          0.9526 |         2.94 |
| Gradient Boosting |           0.5 |          0.9662 |        41.35 |
| XGBoost           |           0.5 |          0.9492 |        38.07 |
| AdaBoost          |           1.5 |          0.9255 |         7.17 |
| LightGBM          |          15   |          0.9662 |         5.06 |
| CatBoost          |           7   |          0.9442 |       122.55 |


## 5. Вопросы (ответить текстом в следующей ячейке)

1. Какой алгоритм показал максимальную точность? Какие параметры к этому привели?

2. Какой алгоритм быстрее всего обучался? Во сколько раз он быстрее самого медленного?

3. У каких алгоритмов наблюдалось переобучение? При каких параметрах?

4. Какой алгоритм вы выбрали бы для продакшн-системы с миллионом текстов? Почему?

# Напишите свои ответы здесь

**Ответ 1:**
Максимальное значение accuracy было достигнуто с помощью алгоритмов **Gradient Boosting при learning_rate=0.5** и **LightGBM при num_leaves=15**. Однако **лучшим** можно считать **LightGBM**, который обучается за **5.06 sec**.

**Ответ 2:**
Быстрее всего обучался алгоритм **Decision Tree** (0.72 sec), что **в 170 раз быстрее**, чем алгоритм CatBoost (122.55 sec).

**Ответ 3:**
Переобучение наблюдается у алгоритмов, когда при повышении параметра растет время обучения, но качество не улучшается. Это заметно у алгоритмов **AdaBoost при learning_rate=2** и **LightGBM при num_leaves>15** (скорее всего num_leaves=15 - это оптимальное значение).

**Ответ 4:**
Для продакш-системы с миллионом текстов я бы выбрала самый быстрый алгоритм, который при этом качественно выводит результат, т.е. **LightGBM**.

## Критерии оценки

| Что оценивается | Баллы |
|----------------|-------|
| Заполнены все 7 таблиц (правильно собраны accuracy и время) | 3 |
| Заполнена итоговая таблица лучших результатов | 1 |
| Ответы на 4 вопроса (по 0.5 балла) | 2 |
| Код воспроизводим (random_state=42, порядок ячеек корректен) | 1 |
| **Качество кода** (отсутствие дублирования, осмысленные имена переменных, циклы вместо копипасты, использование .to_markdown()) | 3 |
| **Итого** | **10** |

### Детали по качеству кода (+3 балла):
- **+1** — использование единого шаблона для всех экспериментов (цикл + list.append + pd.DataFrame)
- **+1** — правильное форматирование вывода (округление времени и accuracy до 2-4 знаков)
- **+1** — читаемые имена переменных, комментарии, отсутствие `eval()`, `exec()` и магических чисел

**Штрафы:**
- -1 балл, если код не запускается без ошибок
- -1 балл, если отсутствует `random_state=42` в любой из моделей
- -1 балл, если таблицы выведены криво (не через `to_markdown` или нечитаемый `print`)